# 📈 Every plot type, on real data

**Time:** ~25 minutes &nbsp;•&nbsp; **Goal:** draw each of the four plots you met in the lecture, repeatedly, until `fig, ax = plt.subplots()` is something your fingers do without you.

matplotlib is a large API and you have seen it once. That is not the same as knowing it. This notebook is deliberately repetitive: the same five lines, over and over, with a different gene each time.

The organising idea underneath the repetition is the one that decides which plot you reach for:

> **How many distinct values does this column have?**
>
> A handful of repeated labels — **categorical**. Thousands of different numbers — **continuous**.

| what you have | the plot |
|---|---|
| one continuous column | `ax.hist()` |
| one categorical column | `ax.bar()` / `ax.barh()` |
| categorical **×** continuous | `ax.boxplot()` |
| continuous **×** continuous | `ax.scatter()` |

## Our data — 1,165 cancer cell lines

Researchers took every one of these cell lines and used CRISPR to switch off each gene in turn. Each number is a **dependency score**: how badly the cells did without that gene.

- `0` — the cells did not care
- `−1` — losing that gene hurt
- `−2` and below — losing it was close to lethal

`oncotree_lineage` is the tissue the cancer came from. Everything else in `GENES` is a score.

Run this first.

In [ ]:
# Run this first. Every exercise below uses `df`.
import urllib.request
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab07/data/"

DEPMAP = "depmap.csv"
if not Path(DEPMAP).exists():
    urllib.request.urlretrieve(DATA_URL + DEPMAP, DEPMAP)

df = pd.read_csv(DEPMAP)

GENES = ["SOX10", "BRAF", "KRAS", "NRAS", "PAX8", "MYB", "CTNNB1", "EGFR",
         "MDM2", "ESR1", "RAN", "RPL5", "RPS6", "PSMA1", "SNRPD1", "EIF4A3",
         "A1BG", "AADAC"]

print(df.shape)
df[["cell_line_name", "oncotree_lineage", "SOX10", "BRAF", "RAN"]].head()

## Skill 1 — One continuous column: `ax.hist()`

A histogram takes a column of numbers, chops the range into bins, and counts how many fall in each. It is the only honest way to look at a single continuous column — a mean tells you where the middle is and hides everything else.

The five lines never change:

```python
fig, ax = plt.subplots()
ax.hist(df["SOX10"], bins=40)
ax.set_xlabel("SOX10 dependency score")
ax.set_ylabel("cell lines")
plt.show()
```

In [ ]:
# DEMO - SOX10 across all 1,165 cell lines
fig, ax = plt.subplots()
fig.set_size_inches(7, 3.5)

ax.hist(df["SOX10"], bins=40, color="#5b9df9")

ax.set_xlabel("SOX10 dependency score")
ax.set_ylabel("number of cell lines")
ax.set_title("Most cells do not need SOX10 — but some really do")
plt.show()

In [ ]:
# DEMO - bins is a decision, not a default
fig, axes = plt.subplots(1, 3)
fig.set_size_inches(12, 3)

for ax, n in zip(axes, [5, 40, 300]):
    ax.hist(df["SOX10"], bins=n, color="#5b9df9")
    ax.set_title(f"bins={n}")
    ax.set_xlabel("SOX10")

fig.tight_layout()
plt.show()

Five bins hides the tail. Three hundred turns it into noise. Forty shows you there is a bulk near zero and a small group a long way left — which is the finding. **Always try three bin counts before you believe a histogram.**

### 📈 DIY challenge — same five lines, new gene

1. Draw the histogram for `KRAS`. Label the axes properly.
2. Now `RAN`. Where is the bulk of the distribution, and what does that tell you about the gene?
3. Now `A1BG`. Compare the three by eye — which gene does every cell need, which do only some cells need, and which does nobody need?
4. Put `MDM2` and `EGFR` on the *same* axes with two `ax.hist()` calls and `alpha=0.5`, and add `ax.legend()`.

> 💡 For question 4 you need `label="MDM2"` inside each `ax.hist()` call — `ax.legend()` reads those labels.

In [ ]:
# TODO - your turn

## Skill 2 — One categorical column: `ax.barh()`

For a categorical column, the thing you want is **how many of each**. `value_counts()` does the counting; `ax.barh()` draws it.

```python
counts = df["oncotree_lineage"].value_counts()
fig, ax = plt.subplots()
ax.barh(counts.index, counts.values)
```

`barh` rather than `bar` for anything with text labels: a horizontal bar gives the label a whole line to itself, so nothing has to be rotated 45° and squinted at.

In [ ]:
# DEMO - how many cell lines per lineage?
counts = df["oncotree_lineage"].value_counts()
print("distinct lineages:", len(counts))

fig, ax = plt.subplots()
fig.set_size_inches(7, 7)

ax.barh(counts.index, counts.values, color="#5fb37a")
ax.set_xlabel("number of cell lines")
ax.set_title("All 30 lineages")
plt.show()

Thirty bars, most of them tiny. Readable, but it is a table pretending to be a figure.

**When a categorical column has too many levels, take the top few and be explicit about it.** The half you cut is not hidden if you say so in the title.

In [ ]:
# DEMO - the top ten, and an honest title
top = df["oncotree_lineage"].value_counts().head(10)

fig, ax = plt.subplots()
fig.set_size_inches(7, 3.5)

ax.barh(top.index[::-1], top.values[::-1], color="#5fb37a")
ax.set_xlabel("number of cell lines")
ax.set_title(f"Ten largest lineages ({top.sum()} of {len(df)} cell lines)")
plt.show()

`[::-1]` reverses the order. `value_counts()` gives you largest first, and `barh` draws the first item at the **bottom** — so without the reversal the biggest bar ends up at the bottom of the chart, which reads backwards.

### 📈 DIY challenge

1. Draw the same chart for `oncotree_primary_disease`. How many distinct values are there? Is a bar chart usable at all?
2. Take the top 15 diseases and draw those instead, with the count in the title.
3. Colour the bars so that any lineage with fewer than 50 cell lines is grey and the rest are blue. (Build a list of colours with a list comprehension and pass it to `color=`.)

> 💡 `ax.barh(names, values, color=["#5b9df9", "#9ca3af", ...])` accepts one colour per bar.

In [ ]:
# TODO - your turn

## Skill 3 — Categorical × continuous: `ax.boxplot()`

Now the interesting shape: one continuous column, **split by** a categorical one. A mean per group would give you one number each and hide the spread. A boxplot gives you the median, the quartiles and the outliers, per group.

`boxplot` wants a **list of arrays** — one per group — and it labels them from `tick_labels`:

```python
groups = [df[df["oncotree_lineage"] == name]["SOX10"] for name in names]
ax.boxplot(groups, tick_labels=names)
```

That list comprehension is the fiddly part, and it is worth reading slowly. For each lineage name, pull the rows with that lineage, take the SOX10 column. One array per group, in the same order as `names`.

In [ ]:
# DEMO - SOX10, split by lineage
names = ["Skin", "Lung", "Lymphoid", "Breast", "CNS/Brain"]
groups = [df[df["oncotree_lineage"] == n]["SOX10"] for n in names]

fig, ax = plt.subplots()
fig.set_size_inches(7, 3.5)

ax.boxplot(groups, tick_labels=names)
ax.axhline(0, color="#9ca3af", linewidth=0.8)

ax.set_ylabel("SOX10 dependency")
ax.set_title("Only melanoma needs SOX10")
plt.show()

One group sits well below the rest and the other four sit on top of each other around zero. That is the whole result, and no table of means would have shown you how *separated* the skin lines are.

`ax.axhline(0, ...)` draws the reference line at "no effect". On a dependency plot it should be there every time — without it, the reader has to find zero on the axis for themselves.

### 📈 DIY challenge — same plot, new marker gene

1. `MYB` across `Lymphoid`, `Myeloid`, `Lung`, `Skin`. Which lineages need it?
2. `PAX8` across `Ovary/Fallopian Tube`, `Kidney`, `Lung`, `Skin`.
3. `EGFR` across `Head and Neck`, `Lung`, `Lymphoid`, `Skin`.
4. Now a control: `A1BG` across any four lineages. What does a boxplot of a gene nobody needs look like?

> 💡 You are changing two things each time — the gene name and the list of lineages. Everything else is copy-paste. That is the point.

In [ ]:
# TODO - your turn

## Skill 4 — Continuous × continuous: `ax.scatter()`

Two numeric columns, one point per cell line.

```python
ax.scatter(df["BRAF"], df["SOX10"], s=8, alpha=0.3)
```

With 1,165 points, `s=` (size) and `alpha=` (transparency) are not decoration — at full size and opacity the middle of the cloud is a solid block and you cannot see where the density is.

In [ ]:
# DEMO - do cells that need BRAF also need SOX10?
fig, ax = plt.subplots()
fig.set_size_inches(5, 5)

ax.scatter(df["BRAF"], df["SOX10"], s=8, alpha=0.3, color="#5b9df9")
ax.axhline(0, color="#9ca3af", linewidth=0.8)
ax.axvline(0, color="#9ca3af", linewidth=0.8)

ax.set_xlabel("BRAF dependency")
ax.set_ylabel("SOX10 dependency")
ax.set_title(f"r = {df['BRAF'].corr(df['SOX10']):.2f}")
plt.show()

In [ ]:
# DEMO - a third column, as colour
skin = df[df["oncotree_lineage"] == "Skin"]
rest = df[df["oncotree_lineage"] != "Skin"]

fig, ax = plt.subplots()
fig.set_size_inches(5, 5)

ax.scatter(rest["BRAF"], rest["SOX10"], s=8, alpha=0.3,
           color="#9ca3af", label="other lineages")
ax.scatter(skin["BRAF"], skin["SOX10"], s=12, alpha=0.9,
           color="#5b9df9", label="skin (melanoma)")

ax.set_xlabel("BRAF dependency")
ax.set_ylabel("SOX10 dependency")
ax.legend()
plt.show()

Two `ax.scatter()` calls on the same `ax`, each with its own colour and `label=`. That is the whole trick for adding a categorical third dimension, and you will use it constantly.

The correlation is 0.59 across everything — but that is mostly the skin cluster sitting in the bottom-left corner dragging the line. Within skin alone it is 0.65, and among everything else only 0.13.

### 📈 DIY challenge

1. Scatter `RAN` against `RPL5`. Both are genes every cell needs — what shape do you expect, and what do you get?
2. Scatter `A1BG` against `AADAC`. Compare the axis ranges with question 1.
3. Redraw the BRAF/SOX10 scatter highlighting `Lung` instead of `Skin`. Does the pattern survive?
4. Print the correlation for skin only and for everything else, and put both in the title.

> 💡 `df["A"].corr(df["B"])` gives you the number. An f-string with `:.2f` puts it in the title tidily.

In [ ]:
# TODO - your turn

## Skill 5 — 🏆 The shape *is* the result

You have now drawn a lot of histograms. Here is what they were for.

Every gene in this file falls into one of three groups, and **you can tell which from the shape of its distribution alone**:

| shape | meaning | examples |
|---|---|---|
| narrow, centred far left | every cell needs it — a **common essential** | RAN, RPL5, PSMA1 |
| spike at zero, long left tail | **some** lineage needs it — a selective dependency | SOX10, MYB, PAX8 |
| spike at zero, no tail | nobody needs it | A1BG, AADAC |

Drug targets live in the middle row. The first row would kill the patient too; the third does nothing.

In [ ]:
# DEMO - three genes, three shapes, one figure
fig, axes = plt.subplots(1, 3)
fig.set_size_inches(12, 3)

for ax, gene, colour in zip(axes, ["RAN", "SOX10", "A1BG"],
                            ["#f87171", "#5b9df9", "#9ca3af"]):
    ax.hist(df[gene], bins=40, color=colour)
    ax.axvline(0, color="#374151", linewidth=0.8)
    ax.set_xlim(-5.5, 1)
    ax.set_xlabel(f"{gene} dependency")

axes[0].set_ylabel("cell lines")
fig.suptitle("Essential, selective, irrelevant")
fig.tight_layout()
plt.show()

`ax.set_xlim(-5.5, 1)` is doing essential work here. Without it each panel gets its own axis range and the three look roughly similar — the whole point is that RAN's distribution sits somewhere A1BG's never reaches. **Panels you want compared must share a scale.**

### 🏆 DIY finale — sort the genes yourself

`GENES` holds all 18 columns. Without looking at the table above:

1. Draw a histogram for every gene in `GENES`, in a grid. `fig, axes = plt.subplots(6, 3)` gives you 18 panels; `axes.flat` lets you loop over them in one go. Give every panel the same `set_xlim`.
2. From the shapes alone, sort the 18 genes into the three groups.
3. Check yourself with numbers: print each gene's `mean()` and `skew()`. Which statistic separates the selective genes from the irrelevant ones, and why is it *not* the mean?
4. Pick the gene with the most extreme skew and draw a boxplot of it across six lineages. Does a lineage stand out?

> 💡 Question 3 is the real finding. A selective gene and an irrelevant gene have almost the same mean — around −0.2. What differs is the tail.

In [ ]:
# TODO - your turn

## 🔭 Explore further

- The matplotlib **plot types** gallery — every chart in the library, with code: <https://matplotlib.org/stable/plot_types/>
- The **Anatomy of a figure** diagram, worth pinning above your desk: <https://matplotlib.org/stable/gallery/showcase/anatomy.html>
- Explore these genes yourself at <https://depmap.org/portal/>

## Recap

You can now, without looking anything up:

- draw a histogram of one continuous column, and choose `bins` deliberately
- count a categorical column and draw it with `barh`, top-N when there are too many levels
- split a continuous column by a categorical one with `boxplot` and a list comprehension
- scatter two continuous columns, with `s=` and `alpha=` for 1,165 points
- add a categorical third dimension by calling `scatter` twice with different colours
- read a gene's biology off the **shape** of its distribution

**Next:** `02_graphic_tools` — everything that turns a plot into a figure.